# Pokémon Type Prediction — Supervised Learning

**Mål:** Förutsäga en Pokémons primära typ (`type_1`) utifrån numeriska stats, kategoriska egenskaper och sprite-bilder med hjälp av supervised learning.

Datasetet innehåller 1025 Pokémon med 18 unika typer (Water, Normal, Grass, …). Vi jämför tre algoritmfamiljer — logistisk regression, XGBoost och Random Forest — och undersöker om sprite-bilder (PCA-komprimerade pixelvektorer) tillför prediktivt värde utöver tabellbaserade features.

**Pipeline-översikt:**
1. Datainsamling → 2. Datarensning → 3. EDA → 4. Feature engineering → 5. PCA & split → 6. Modellträning → 7. Utvärdering

---

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, ParameterGrid
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.inspection import permutation_importance
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
%matplotlib inline

# Projektets egna moduler
from settings import (
    DATA_PATH, IMAGE_DIR, PCA_VARIANCE, RANDOM_STATE, TEST_RATIO,
    LR_MAX_ITER, GRID_SEARCH_CV, PERMUTATION_REPEATS, XGB_PARAM_GRID,
    RF_ESTIMATORS, RF_MAX_DEPTH, RF_MIN_SAMPLES_LEAF,
    IMG_SIZE, CHANNELS, IMAGE_PCA_VARIANCE,
)

print("Alla beroenden laddade.")

## 1. Datainsamling

Datasetet (`pokemon_complete.csv`) är semikolonseparerat och innehåller statistik, typ, färg, habitat och sprite-URL för alla Pokémon.

In [ ]:
from load_data import load_data

df_raw = load_data(DATA_PATH)
print(f"\nRader: {df_raw.shape[0]}, Kolumner: {df_raw.shape[1]}")
print(f"Saknade värden totalt: {df_raw.isna().sum().sum()}")
print(f"Dubbletter: {df_raw.duplicated().sum()}")
df_raw.head()

## 2. Datarensning

Rensningen hanterar saknade värden (median-fill för numeriska, "unknown" för kategoriska), konverterar generation-strängar till heltal, extraherar abilities/egg_groups och formaterar `type_1` till heltal.

In [ ]:
from data_clean import clean_data

df_reference = clean_data(df_raw)
print(f"\nEfter rensning: {df_reference.shape[0]} rader, {df_reference.shape[1]} kolumner")
print(f"Saknade värden: {df_reference.isna().sum().sum()}")
df_reference.head()

### Fördelning av type_1 (målvariabel)

Klassbalansen är viktig — ovanliga typer (Flying, Fairy) är svårare att prediktera.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
type_counts = df_reference["type_1"].value_counts().sort_values()
type_counts.plot(kind="barh", ax=ax, color="#4C78A8")
ax.set_title("Fördelning av type_1 (målvariabel)")
ax.set_xlabel("Antal Pokémon")
ax.set_ylabel("")
for i, (count, name) in enumerate(zip(type_counts.values, type_counts.index)):
    ax.text(count + 2, i, str(count), va="center", fontsize=9)
fig.tight_layout()
plt.show()

print(f"Antal unika typer: {df_reference['type_1'].nunique()}")
print(f"Vanligaste typ: {type_counts.index[-1]} ({type_counts.values[-1]} st)")
print(f"Ovanligaste typ: {type_counts.index[0]} ({type_counts.values[0]} st)")

### Korrelationsheatmap — numeriska battle stats

In [ ]:
BATTLE_STATS = ["hp", "attack", "defense", "sp_attack", "sp_defense", "speed"]
EXTRA_NUMERIC = ["height_m", "weight_kg", "capture_rate"]
numeric_cols = BATTLE_STATS + EXTRA_NUMERIC

fig, ax = plt.subplots(figsize=(10, 8))
corr = df_reference[numeric_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Korrelationer mellan numeriska features")
fig.tight_layout()
plt.show()

### Saknade värden — före och efter rensning

In [ ]:
raw_missing = df_raw.isna().sum()
raw_missing = raw_missing[raw_missing > 0].sort_values(ascending=False)
cleaned_missing = df_reference.isna().sum()
cleaned_missing = cleaned_missing[cleaned_missing > 0].sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if not raw_missing.empty:
    raw_missing.head(15).sort_values().plot(kind="barh", ax=axes[0], color="#F58518")
    axes[0].set_title(f"Saknade värden — Rådata (totalt {raw_missing.sum()})")
else:
    axes[0].text(0.5, 0.5, "Inga saknade värden", ha="center", va="center")
    axes[0].set_title("Saknade värden — Rådata")

if not cleaned_missing.empty:
    cleaned_missing.head(15).sort_values().plot(kind="barh", ax=axes[1], color="#54A24B")
    axes[1].set_title(f"Saknade värden — Rensad data (totalt {cleaned_missing.sum()})")
else:
    axes[1].text(0.5, 0.5, "Inga saknade värden", ha="center", va="center")
    axes[1].set_title("Saknade värden — Rensad data (0)")

fig.tight_layout()
plt.show()

## 3. Feature engineering & träningsdata

Träningsdatan skapas genom att:
- Ta bort identitets-/referenskolumner (namn, pokedex_number, type_2, abilities, …)
- Dummy-koda kategoriska features: `color`, `shape`, `habitat`, `growth_rate`
- Behålla numeriska features: hp, attack, defense, sp_attack, sp_defense, speed, height_m, weight_kg, capture_rate

In [ ]:
from data_clean import build_training_dataframe

df_training = build_training_dataframe(df_reference)

target_col = df_training.attrs["target_column"]
feature_cols = df_training.attrs["feature_columns"]
target_map = df_training.attrs.get("target_map", {})
target_labels = {int(v): k for k, v in target_map.items()}

print(f"\nAntal features: {len(feature_cols)}")
print(f"Målkolumn: {target_col}")
print(f"\nFeature-lista:")
for i, col in enumerate(feature_cols, 1):
    print(f"  {i:02d}. {col}")

## 4. PCA & tränings/test-split

Datan delas upp 80/20 med stratifiering på `type_1`. Därefter standardiseras features och PCA appliceras (behåller 95 % av variansen). PCA används av logistisk regression för att reducera dimensionalitet och multikollinearitet.

In [ ]:
X = df_training[feature_cols].astype(float)
y = df_training[target_col].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_RATIO,
    stratify=y,
    random_state=RANDOM_STATE,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

pca = PCA(n_components=PCA_VARIANCE, random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"Träningsdata: {X_train.shape[0]} rader")
print(f"Testdata:     {X_test.shape[0]} rader")
print(f"Features före PCA:  {X_train.shape[1]}")
print(f"Features efter PCA: {X_train_pca.shape[1]}")
print(f"Bevarad varians:    {pca.explained_variance_ratio_.sum():.2%}")

### Kumulativ förklarad varians

In [ ]:
cumulative_var = np.cumsum(pca.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, len(cumulative_var) + 1), cumulative_var, "o-", linewidth=2, markersize=4)
ax.axhline(PCA_VARIANCE, color="red", linestyle="--", linewidth=1, label=f"Tröskel ({PCA_VARIANCE:.0%})")
ax.set_title("Kumulativ förklarad varians efter PCA")
ax.set_xlabel("Antal PCA-komponenter")
ax.set_ylabel("Kumulativ förklarad varians")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

print(f"Komponenter som behövs för {PCA_VARIANCE:.0%} varians: {len(cumulative_var)}")

## 5. Modellträning — de tre bästa modellerna

Baserat på den fullständiga körningen (11 modeller) väljer vi ut de tre med bäst resultat:

| Modell | Accuracy | Balanced Accuracy | Macro F1 |
|--------|----------|-------------------|----------|
| **Logistic Regression (PCA)** | 0.488 | 0.405 | 0.404 |
| **Random Forest** | 0.488 | 0.378 | 0.365 |
| **XGBoost stripped features** | 0.444 | 0.420 | 0.404 |

Vi tränar om dessa tre nedan och visualiserar resultaten.

### 5a. Logistisk regression (PCA)

Multinomial logistisk regression tränas på de PCA-transformerade features. Modellen är enkel men ger bra balanced accuracy tack vare att PCA och StandardScaler jämnar ut feature-skalan samt reducerar dimensioner.

In [ ]:
logreg = LogisticRegression(
    solver="lbfgs",
    max_iter=LR_MAX_ITER,
    random_state=RANDOM_STATE,
)
logreg.fit(X_train_pca, y_train)
y_pred_logreg = logreg.predict(X_test_pca)

acc = accuracy_score(y_test, y_pred_logreg)
bal_acc = balanced_accuracy_score(y_test, y_pred_logreg)
f1_macro = f1_score(y_test, y_pred_logreg, average="macro", zero_division=0)

print(f"Logistic Regression (PCA)")
print(f"  Accuracy:          {acc:.3f}")
print(f"  Balanced Accuracy: {bal_acc:.3f}")
print(f"  Macro F1:          {f1_macro:.3f}")

class_labels = np.array(sorted(y_test.unique()))
class_names = [target_labels.get(int(l), str(l)) for l in class_labels]
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_logreg, labels=class_labels, target_names=class_names, zero_division=0))

In [ ]:
# Confusion matrix — Logistic Regression
cm_logreg = confusion_matrix(y_test, y_pred_logreg, labels=class_labels)
cm_df = pd.DataFrame(cm_logreg, index=class_names, columns=class_names)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_title("Confusion Matrix — Logistic Regression (PCA)")
ax.set_xlabel("Predikterad typ")
ax.set_ylabel("Faktisk typ")
fig.tight_layout()
plt.show()

### 5b. Random Forest

Random Forest med 300 träd och balanserade klassvikter. Ger bra accuracy och feature importance direkt.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=RF_ESTIMATORS,
    max_depth=RF_MAX_DEPTH,
    min_samples_leaf=RF_MIN_SAMPLES_LEAF,
    class_weight="balanced",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

acc = accuracy_score(y_test, y_pred_rf)
bal_acc = balanced_accuracy_score(y_test, y_pred_rf)
f1_macro = f1_score(y_test, y_pred_rf, average="macro", zero_division=0)

print(f"Random Forest")
print(f"  Accuracy:          {acc:.3f}")
print(f"  Balanced Accuracy: {bal_acc:.3f}")
print(f"  Macro F1:          {f1_macro:.3f}")

print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_rf, labels=class_labels, target_names=class_names, zero_division=0))

In [ ]:
# Confusion matrix — Random Forest
cm_rf = confusion_matrix(y_test, y_pred_rf, labels=class_labels)
cm_rf_df = pd.DataFrame(cm_rf, index=class_names, columns=class_names)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm_rf_df, annot=True, fmt="d", cmap="Greens", ax=ax)
ax.set_title("Confusion Matrix — Random Forest")
ax.set_xlabel("Predikterad typ")
ax.set_ylabel("Faktisk typ")
fig.tight_layout()
plt.show()

In [ ]:
# Feature importance — Random Forest
importance_rf = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf.feature_importances_,
}).sort_values("importance", ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))
top20 = importance_rf.head(20).sort_values("importance", ascending=True)
ax.barh(top20["feature"], top20["importance"], color="#54A24B")
ax.set_title("Feature Importance — Random Forest (topp 20)")
ax.set_xlabel("Importance")
fig.tight_layout()
plt.show()

In [ ]:
# Permutation importance — Random Forest
print("Beräknar permutation importance (kan ta en stund)...")
perm_rf = permutation_importance(
    rf, X_test, y_test,
    n_repeats=PERMUTATION_REPEATS,
    random_state=RANDOM_STATE,
    scoring="balanced_accuracy",
    n_jobs=-1,
)
perm_rf_df = pd.DataFrame({
    "feature": feature_cols,
    "importance_mean": perm_rf.importances_mean,
}).sort_values("importance_mean", ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))
top20_perm = perm_rf_df.head(20).sort_values("importance_mean", ascending=True)
ax.barh(top20_perm["feature"], top20_perm["importance_mean"], color="#2E86AB")
ax.set_title("Permutation Importance — Random Forest (topp 20)")
ax.set_xlabel("Mean importance (balanced accuracy)")
fig.tight_layout()
plt.show()

### 5c. XGBoost — stripped features

XGBoost tränas enbart på dummy-kodade kategorier (`color`, `shape`, `habitat`, `growth_rate`) plus `sp_attack`. Denna modell testar hypotesen att typinformation främst bärs av icke-numeriska egenskaper.

In [ ]:
# Definiera stripped features
DUMMY_SOURCE_COLUMNS = ["color", "shape", "habitat", "growth_rate"]
STRIPPED_FEATURE_PREFIXES = tuple(f"{col}_" for col in DUMMY_SOURCE_COLUMNS)
STRIPPED_NUMERIC_FEATURES = ["sp_attack"]

stripped_cols = [
    col for col in feature_cols
    if col in STRIPPED_NUMERIC_FEATURES
    or col.startswith(STRIPPED_FEATURE_PREFIXES)
]
print(f"Stripped features: {len(stripped_cols)} st")
for i, col in enumerate(stripped_cols, 1):
    print(f"  {i:02d}. {col}")

In [ ]:
X_train_stripped = X_train[stripped_cols]
X_test_stripped = X_test[stripped_cols]

sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)

base_xgb = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    num_class=len(np.unique(y_train)),
    tree_method="hist",
    n_jobs=1,
    random_state=RANDOM_STATE,
)

grid_search = GridSearchCV(
    estimator=base_xgb,
    param_grid=XGB_PARAM_GRID,
    scoring="balanced_accuracy",
    cv=GRID_SEARCH_CV,
    n_jobs=-1,
    refit=True,
    verbose=0,
)

print(f"Kör GridSearchCV med {len(list(ParameterGrid(XGB_PARAM_GRID)))} kombinationer × {GRID_SEARCH_CV} folds...")
grid_search.fit(X_train_stripped, y_train, sample_weight=sample_weight)
xgb_stripped = grid_search.best_estimator_
y_pred_xgb = xgb_stripped.predict(X_test_stripped)

acc = accuracy_score(y_test, y_pred_xgb)
bal_acc = balanced_accuracy_score(y_test, y_pred_xgb)
f1_macro = f1_score(y_test, y_pred_xgb, average="macro", zero_division=0)

print(f"\nXGBoost Stripped Features")
print(f"  Accuracy:          {acc:.3f}")
print(f"  Balanced Accuracy: {bal_acc:.3f}")
print(f"  Macro F1:          {f1_macro:.3f}")
print(f"  Bästa parametrar:  {grid_search.best_params_}")
print(f"  Bästa CV score:    {grid_search.best_score_:.3f}")

print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_xgb, labels=class_labels, target_names=class_names, zero_division=0))

In [ ]:
# Confusion matrix — XGBoost stripped
cm_xgb = confusion_matrix(y_test, y_pred_xgb, labels=class_labels)
cm_xgb_df = pd.DataFrame(cm_xgb, index=class_names, columns=class_names)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm_xgb_df, annot=True, fmt="d", cmap="Oranges", ax=ax)
ax.set_title("Confusion Matrix — XGBoost Stripped Features")
ax.set_xlabel("Predikterad typ")
ax.set_ylabel("Faktisk typ")
fig.tight_layout()
plt.show()

In [ ]:
# Feature importance — XGBoost stripped
importance_xgb = pd.DataFrame({
    "feature": stripped_cols,
    "importance": xgb_stripped.feature_importances_,
}).sort_values("importance", ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))
plot_data = importance_xgb.head(20).sort_values("importance", ascending=True)
ax.barh(plot_data["feature"], plot_data["importance"], color="#F58518")
ax.set_title("Feature Importance — XGBoost Stripped (topp 20)")
ax.set_xlabel("Importance")
fig.tight_layout()
plt.show()

## 6. Fullständig modelljämförelse — alla 11 modeller

Tabellen nedan visar resultaten från den fullständiga körningen som inkluderade alla 11 modellvarianter, inklusive de som använde PCA-komprimerade sprite-bilder.

In [ ]:
# Resultat från den fullständiga körningen (app.py)
full_comparison = pd.DataFrame([
    {"model": "logreg_pca",                             "accuracy": 0.488, "balanced_accuracy": 0.405, "macro_f1": 0.404, "weighted_f1": 0.467},
    {"model": "xgboost_grid_search",                    "accuracy": 0.463, "balanced_accuracy": 0.387, "macro_f1": 0.375, "weighted_f1": 0.470},
    {"model": "xgboost_stripped_features",              "accuracy": 0.444, "balanced_accuracy": 0.420, "macro_f1": 0.404, "weighted_f1": 0.451},
    {"model": "logreg_pca_with_images",                 "accuracy": 0.356, "balanced_accuracy": 0.270, "macro_f1": 0.267, "weighted_f1": 0.342},
    {"model": "xgboost_grid_search_with_images",        "accuracy": 0.351, "balanced_accuracy": 0.217, "macro_f1": 0.198, "weighted_f1": 0.309},
    {"model": "xgboost_stripped_features_with_images",  "accuracy": 0.327, "balanced_accuracy": 0.201, "macro_f1": 0.182, "weighted_f1": 0.282},
    {"model": "random_forest",                          "accuracy": 0.488, "balanced_accuracy": 0.378, "macro_f1": 0.365, "weighted_f1": 0.455},
    {"model": "random_forest_with_images",              "accuracy": 0.371, "balanced_accuracy": 0.213, "macro_f1": 0.182, "weighted_f1": 0.286},
    {"model": "logreg_image_only",                      "accuracy": 0.146, "balanced_accuracy": 0.091, "macro_f1": 0.084, "weighted_f1": 0.133},
    {"model": "xgboost_image_only",                     "accuracy": 0.171, "balanced_accuracy": 0.100, "macro_f1": 0.091, "weighted_f1": 0.146},
    {"model": "random_forest_image_only",               "accuracy": 0.195, "balanced_accuracy": 0.097, "macro_f1": 0.063, "weighted_f1": 0.115},
])

# Lägg till grupperingsflagga
def model_group(name):
    if "image_only" in name:
        return "Enbart bilder"
    elif "with_images" in name:
        return "Tabell + bilder"
    else:
        return "Enbart tabell"

full_comparison["grupp"] = full_comparison["model"].apply(model_group)
print(full_comparison.to_string(index=False))

In [ ]:
# Modelljämförelse — barplot
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

colors = {"Enbart tabell": "#4C78A8", "Tabell + bilder": "#F58518", "Enbart bilder": "#E45756"}

# Accuracy & Balanced Accuracy
for ax, metric, title in zip(
    axes,
    ["accuracy", "balanced_accuracy"],
    ["Accuracy", "Balanced Accuracy"],
):
    sorted_df = full_comparison.sort_values(metric, ascending=True)
    bar_colors = [colors[g] for g in sorted_df["grupp"]]
    ax.barh(sorted_df["model"], sorted_df[metric], color=bar_colors)
    ax.set_title(title)
    ax.set_xlim(0, 0.6)
    ax.axvline(1/18, color="gray", linestyle=":", alpha=0.7, label="Slumpchans (1/18)")
    ax.legend(fontsize=8)

fig.suptitle("Modelljämförelse — alla 11 varianter", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

# Färgförklaring
print("Färgkodning:")
print("  Blå   = Enbart tabellfeatures")
print("  Orange = Tabell + bildfeatures")
print("  Röd   = Enbart bildfeatures")

In [ ]:
# Jämförelse per grupp — sammanfattning
group_summary = full_comparison.groupby("grupp")[["accuracy", "balanced_accuracy", "macro_f1"]].mean()
group_summary = group_summary.reindex(["Enbart tabell", "Tabell + bilder", "Enbart bilder"])

fig, ax = plt.subplots(figsize=(10, 5))
group_summary.plot(kind="bar", ax=ax, color=["#4C78A8", "#2E86AB", "#54A24B"])
ax.set_title("Genomsnittlig prestanda per feature-grupp")
ax.set_ylabel("Score")
ax.set_ylim(0, 0.6)
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Metric")
fig.tight_layout()
plt.show()

print("\nSlutsats: Bildfeatures försämrar konsekvent resultaten.")
print("Tabellbaserade modeller når ~45-49% accuracy, medan bildmodeller ligger på 15-20%.")

## 7. Slutsatser

### Resultat
- **Bästa accuracy (0.488):** Logistic Regression (PCA) och Random Forest delar toppositionen.
- **Bästa balanced accuracy (0.420):** XGBoost med stripped features — denna modell hanterar klassimbalansen bäst.
- **Bästa macro F1 (0.404):** Delat mellan Logistic Regression (PCA) och XGBoost stripped.

### Bildfeatures tillför inget värde
Alla modellvarianter med sprite-bilder (PCA-komprimerade pixelvektorer) presterar *sämre* än sina tabellbaserade motsvarigheter. Image-only-modellerna ligger nära slumpchans. Sprite-bilderna verkar främst fånga visuell likhet snarare än typinformation — en blå Water-typ och en blå Flying-typ kan därför bli svåra att skilja åt med enkla pixelvektorer.

### Viktigaste features
Random Forest och XGBoost pekar konsekvent ut **color**, **shape**, **habitat** och **growth_rate** som de starkaste prediktörerna, medan rena battle stats (hp, attack, etc.) bidrar mindre. Detta stämmer med domänkunskap — en Pokémons typ hänger samman med dess ekologiska egenskaper snarare än stridsstyrka.

### Problemets svårighetsgrad
18 klasser med ojämn fördelning (Water: ~130 st, Flying: ~9 st) gör detta till ett svårt klassificeringsproblem. ~49% accuracy mot 5.6% slumpchans visar att modellerna lär sig meningsfulla mönster, men det finns en tydlig övre gräns givet de tillgängliga features.